# ShinyySwapper - Professional Face Swap Model Training
## Optimized for Google Colab Free GPU

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q torch torchvision opencv-python insightface onnxruntime-gpu albumentations kornia lpips facexlib gfpgan gdown

## Download 80K Training Images (FFHQ + CelebA-HQ)

In [ ]:
import gdown
import zipfile
import os
from pathlib import Path
import shutil
from tqdm import tqdm

output_dir = '/content/drive/MyDrive/shinyyswapper_data'
os.makedirs(output_dir, exist_ok=True)

# Download FFHQ (70k images)
print('📥 Downloading FFHQ 70k images...')
gdown.download_folder('https://drive.google.com/drive/folders/1tZUcXDBeOibC6jcMCtgRRz67pzrAHeHL', 
                      output=output_dir, quiet=False, remaining_ok=True, use_cookies=False)

# Download CelebA-HQ subset (10k images)
print('📥 Downloading CelebA-HQ 10k images...')
celeba_url = 'https://drive.google.com/uc?id=1badu11NqxGf6qM3PTTooQDJvQbejgbTv'
gdown.download(celeba_url, '/content/celeba.zip', quiet=False)

with zipfile.ZipFile('/content/celeba.zip', 'r') as z:
    z.extractall('/content/celeba_temp')

# Copy first 10k images
celeba_imgs = sorted(list(Path('/content/celeba_temp').rglob('*.jpg')))[:10000]
for img in tqdm(celeba_imgs, desc='Copying CelebA'):
    shutil.copy(img, output_dir)

shutil.rmtree('/content/celeba_temp')
os.remove('/content/celeba.zip')

total = len(list(Path(output_dir).rglob('*.jpg'))) + len(list(Path(output_dir).rglob('*.png')))
print(f'✅ Dataset ready: {total} images')

In [ ]:
# Clone your repo or upload files
!git clone YOUR_REPO_URL /content/shinyyswapper
%cd /content/shinyyswapper

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Start training on 80k images (auto-resumes from last checkpoint)
!python train.py

## Export Model for Inference

In [ ]:
import torch
from model import Generator

# Load trained model (80k images, 100 epochs)
G = Generator()
checkpoint = torch.load('/content/drive/MyDrive/shinyyswapper_checkpoints/checkpoint_epoch_99.pt')
G.load_state_dict(checkpoint['G_state'])
G.eval()

# Export to ONNX for fast CPU inference
dummy_img = torch.randn(1, 3, 256, 256)
dummy_id = torch.randn(1, 512)

torch.onnx.export(
    G,
    (dummy_img, dummy_id),
    '/content/drive/MyDrive/shinyyswapper_model.onnx',
    input_names=['target_image', 'source_identity'],
    output_names=['swapped_face'],
    dynamic_axes={'target_image': {0: 'batch'}, 'source_identity': {0: 'batch'}, 'swapped_face': {0: 'batch'}}
)

print("ShinyySwapper model exported successfully!")